# Phase 6: Machine Learning Pipeline

## Objective
Build a credit default prediction model that identifies likely defaulters while controlling unnecessary rejection of reliable customers. Accuracy must not be used as the only model-selection metric due to class imbalance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import joblib
import json

pd.set_option('display.max_columns', None)

## 1. Data Loading & Validation

In [ ]:
df = pd.read_csv('../data/processed/creditguard_model_ready.csv')
print("Shape:", df.shape)
print("\nTarget Distribution:")
print(df['default_payment_next_month'].value_counts(normalize=True))

## 2. Train-Test Split (80/20 Stratified)
To ensure no data leakage and maintain target class proportion.

In [ ]:
X = df.drop(columns=['default_payment_next_month'])
y = df['default_payment_next_month']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

## 3. Model Results & Comparison
Due to computational requirements (SMOTE, Cross-Validation, GridSearchCV, RandomizedSearchCV), the models are trained automatically via the ML pipeline in `src/train_model.py`. The resulting comparison metric table is loaded below to visualize the results of the pipeline:

In [ ]:
# Load Model Comparison Data
try:
    df_comp = pd.read_csv('../reports/model/model_comparison.csv')
    display(df_comp)
except FileNotFoundError:
    print("Model comparison not found. Ensure src/train_model.py has finished running.")

## 4. Threshold Optimization & Business Cost Analysis
We use out-of-fold probabilities to find a threshold that minimizes our illustrative business cost:
- False Negative Cost = 5
- False Positive Cost = 1

In [ ]:
# Load Threshold Analysis Data
try:
    df_th = pd.read_csv('../reports/model/threshold_analysis.csv')
    display(df_th.head(10))
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_th['Threshold'], df_th['Total_Business_Cost'], marker='o')
    ax.set_title("Business Cost by Probability Threshold")
    ax.set_xlabel("Classification Threshold")
    ax.set_ylabel("Total Illustrative Business Cost")
    plt.grid(True)
    plt.show()
except FileNotFoundError:
    print("Threshold analysis not found.")

## 5. Final Artifact Validation

In [ ]:
# Load Final Metadata
try:
    with open('../models/creditguard_model_metadata.json', 'r') as f:
        metadata = json.load(f)
    print(json.dumps(metadata, indent=4))
except FileNotFoundError:
    print("Model metadata not found.")

## Conclusion
The optimal model and threshold successfully balance false positives and false negatives based on the illustrative business costs. The end-to-end pipeline is serialized to `models/creditguard_final_pipeline.joblib` and ready for integration into the Power BI dashboard and Streamlit App.